# XGBoost (with Optuna tuning)

In [2]:
import os, re, glob
import numpy as np
import pandas as pd
import optuna
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb
from tqdm import tqdm
import warnings
from statsmodels.tsa.seasonal import STL
import holidays
warnings.filterwarnings('ignore')

In [3]:
train = pd.read_csv('./data/train/train.csv').copy()

In [4]:
def feature_engineering(df, weather_df=None):
    import numpy as np, pandas as pd

    # 1) 기본 전처리
    df[['영업장명', '메뉴명']] = df['영업장명_메뉴명'].str.extract(r'^([^_ ]+)[_ ](.+)$')
    df['영업일자'] = pd.to_datetime(df['영업일자'], errors='coerce')
    df['month'] = df['영업일자'].dt.month
    df['weekday'] = df['영업일자'].dt.weekday
    df['day_of_year'] = df['영업일자'].dt.dayofyear
    df['is_weekend'] = (df['weekday'] >= 5).astype('int8')

    df['days_in_month']     = df['영업일자'].dt.days_in_month
    df['dom_frac']          = (df['영업일자'].dt.day / df['days_in_month']).astype('float32')  # 0~1
    df['days_to_month_end'] = (df['days_in_month'] - df['영업일자'].dt.day).astype('int16')
    df['is_month_start']    = df['영업일자'].dt.is_month_start.astype('int8')
    df['is_month_end']      = df['영업일자'].dt.is_month_end.astype('int8')
    df['week_of_month']     = ((df['영업일자'].dt.day - 1) // 7).astype('int8')

    # 2) 주기/사인코사인
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

    # ISO 주차(연/주)
    iso = df['영업일자'].dt.isocalendar()
    df['iso_year'] = iso.year.astype(int)
    df['iso_week'] = iso.week.astype(int)
    # 문자열 키가 필요하면:
    df['year_week'] = (df['iso_year'].astype(str) + '-' + df['iso_week'].astype(str).str.zfill(2))

    # 3) 롤링/래그
    key = '영업장명_메뉴명'
    ycol = '매출수량'
    g = df.groupby(key)[ycol]

    for win in [2,3,5,7]:
        df[f'rolling_avg_{win}d'] = g.apply(lambda s: s.shift(1).rolling(win, min_periods=1).mean()).reset_index(level=0, drop=True)
        df[f'rolling_std_{win}d'] = g.apply(lambda s: s.shift(1).rolling(win, min_periods=1).std()).reset_index(level=0, drop=True)
        df[f'rolling_sum_{win}d'] = g.apply(lambda s: s.shift(1).rolling(win, min_periods=1).sum()).reset_index(level=0, drop=True)

    for lag in range(1,8):
        df[f'sales_lag_{lag}'] = g.shift(lag)

    # 3-1) 7일 EWM, 기울기, 분위수(모두 shift(1) 기반 → 누수 방지)
    df['ewm_mean_7'] = g.apply(lambda s: s.shift(1).ewm(span=7, adjust=False).mean()).reset_index(level=0, drop=True)

    def _roll_slope(a):
        n = len(a)
        if n <= 1: return 0.0
        x = np.arange(n, dtype=np.float32)
        y = a.astype(np.float32)
        sx, sy = x.sum(), y.sum()
        sxx, sxy = (x*x).sum(), (x*y).sum()
        denom = n*sxx - sx*sx
        return 0.0 if denom == 0 else (n*sxy - sx*sy)/denom

    df['slope_log1p_7'] = g.apply(
        lambda s: np.log1p(s.shift(1)).rolling(7, min_periods=2).apply(_roll_slope, raw=True)
    ).reset_index(level=0, drop=True)

    q10 = g.apply(lambda s: s.shift(1).rolling(7, min_periods=1).quantile(0.10)).reset_index(level=0, drop=True)
    q90 = g.apply(lambda s: s.shift(1).rolling(7, min_periods=1).quantile(0.90)).reset_index(level=0, drop=True)
    q25 = g.apply(lambda s: s.shift(1).rolling(7, min_periods=1).quantile(0.25)).reset_index(level=0, drop=True)
    q75 = g.apply(lambda s: s.shift(1).rolling(7, min_periods=1).quantile(0.75)).reset_index(level=0, drop=True)
    df['roll_q10_7'] = q10.values
    df['roll_q90_7'] = q90.values
    df['roll_iqr_7'] = (q75 - q25).values

    # 3-2) z-score(7), 최근 0연속, 마지막 비제로까지 경과일
    roll_mean_7 = df['rolling_avg_7d']
    roll_std_7  = df['rolling_std_7d'].replace(0, np.nan)
    y_shift1 = g.shift(1).reset_index(level=0, drop=True)
    df['zscore_7'] = ((y_shift1 - roll_mean_7) / (roll_std_7 + 1e-6)).replace([np.inf, -np.inf], 0).fillna(0)

    def _days_since_nonzero(s):
        cnt = 0
        out = []
        for v in s:
            cnt = 0 if (v > 0 and pd.notna(v)) else cnt + 1
            out.append(cnt)
        return out
    
    tmp_days = (g.apply(lambda s: pd.Series(_days_since_nonzero(s.shift(1)), index=s.index))
              .reset_index(level=0, drop=True))
    df['days_since_nonzero'] = tmp_days.to_numpy(dtype='int16', copy=False)

    def _zero_streak_last7(series):
        w = series.tail(7).values
        run = 0
        for v in w[::-1]:
            if (pd.isna(v) or v==0): run += 1
            else: break
        return run
    df['zero_streak_7'] = g.apply(
        lambda s: s.shift(1).rolling(7, min_periods=1).apply(lambda x: _zero_streak_last7(pd.Series(x)), raw=False)
    ).reset_index(level=0, drop=True).fillna(0)

    # 4) 누적/주차/월차 통계
    df['cum_sales'] = df.groupby(key)[ycol].cumsum()
    df['weekly_avg_sales'] = df.groupby([key, 'year_week'])[ycol].transform('mean')
    df['weekly_std_sales'] = df.groupby([key, 'year_week'])[ycol].transform('std')
    df['weekly_sum_sales'] = df.groupby([key, 'year_week'])[ycol].transform('sum')
    df['weekly_min_sales'] = df.groupby([key, 'year_week'])[ycol].transform('min')
    df['weekly_max_sales'] = df.groupby([key, 'year_week'])[ycol].transform('max')
    # 누수 줄이려면 평균만 사용하고 shift(1)한 것 위주로 써도 좋음
    df['weekly_avg_shift1'] = df.groupby(key)['weekly_avg_sales'].shift(1)

    # (선택) 원시 캘린더 정수 피처는 과적합 줄이려면 줄여도 됨
    df = df.drop(columns=['day','iso_year','day_of_year','year_week'], errors='ignore')

    return df

In [5]:
# 고정 카테고리(학습/추론 모두 동일하게)
WEEKDAY_CATS = list(range(7))      # 0~6
MONTH_CATS   = list(range(1, 13))  # 1~12

ONEHOT_EXPECTED = [f'wd_{i}' for i in WEEKDAY_CATS] + [f'm_{i}' for i in MONTH_CATS]

def add_calendar_onehots(df: pd.DataFrame, drop_original: bool = True) -> pd.DataFrame:
    out = df.copy()

    # 고정 카테고리로 캐스팅
    out['weekday'] = out['weekday'].astype(pd.CategoricalDtype(categories=WEEKDAY_CATS))
    out['month']   = out['month'].astype(pd.CategoricalDtype(categories=MONTH_CATS))

    d_w = pd.get_dummies(out['weekday'], prefix='wd', dtype='int8')
    d_m = pd.get_dummies(out['month'],   prefix='m',  dtype='int8')

    out = pd.concat([out, d_w, d_m], axis=1)

    if drop_original:
        out = out.drop(columns=['weekday', 'month'], errors='ignore')

    # 누락된 원핫 컬럼 채우기(테스트에서 일부 카테고리 미등장 대비)
    for c in ONEHOT_EXPECTED:
        if c not in out.columns:
            out[c] = np.int8(0)

    return out

In [6]:
train = feature_engineering(train)
train = add_calendar_onehots(train, drop_original=True)

In [7]:
for i in range(10):  # 0부터 9까지
    filename = f"./data/test/TEST_0{i}.csv"
    df = pd.read_csv(filename).copy()
    
    # feature_engineering 함수 적용
    df = feature_engineering(df)
    df = add_calendar_onehots(df, drop_original=True)
    
    # 처리한 데이터 저장
    df.to_csv(f"./data/test_pre/TEST_0{i}.csv", index=False, encoding='utf-8-sig')

In [8]:
# -----------------------------
# Metric & Local Validation
# -----------------------------
DEFAULT_STORE_WEIGHTS = {'담하': 1, '미라시아': 1}

def _smape(a, p, eps=1e-12):
    return 2.0 * np.abs(a - p) / (np.abs(a) + np.abs(p) + eps)

def weighted_smape_df(df: pd.DataFrame,
                      pred_col: str,
                      actual_col: str,
                      store_col: str = '영업장명',
                      item_col: str = '영업장명_메뉴명',
                      store_weights: dict | None = None) -> float:
    if actual_col not in df.columns or pred_col not in df.columns:
        return np.nan
        
    valid = df[df[actual_col] != 0].copy()
    
    if valid.empty:
        return np.nan
    valid['_smape'] = _smape(valid[actual_col].values, valid[pred_col].values)
    total = 0.0
    wsum  = 0.0
    sw = store_weights or {}
    for store, g in valid.groupby(store_col):
        w = sw.get(store, 1.0)
        item_means = g.groupby(item_col)['_smape'].mean()
        if len(item_means) == 0:
            continue
        total += w * item_means.mean()
        wsum  += w
    return (total / wsum) if wsum > 0 else np.nan

In [9]:
def local_validate_on_test(df_test: pd.DataFrame,
                           model,
                           features: list,
                           target_col: str = '매출수량',
                           menu_col: str = '영업장명_메뉴명',
                           store_col: str = '영업장명',
                           date_col: str = '영업일자',
                           store_weights: dict | None = None,
                           fe_done: bool = False,
                           encoded_done: bool = False,
                           clip_to_zero: bool = True) -> float:
    df = df_test.copy()

    # 1) FE/정렬
    if not fe_done:
        df = feature_engineering(df)
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
        df = df.sort_values([menu_col, date_col])

    # 2) 인코딩
    if not encoded_done:
        df = apply_cat_maps(df, cat_maps)

    # 3) 타깃(+1~+7) 생성 (마스크 없이)
    target_cols = []
    for i in range(1, 8):
        col = f'{target_col}_tplus_{i}'
        df[col] = df.groupby(menu_col, sort=False)[target_col].shift(-i)  # fillna 안 함
        target_cols.append(col)
    
    # 4) 검증에 사용할 행 필터링: 7개 타깃 모두 존재하고 모두 > 0
    df = df.loc[df[target_cols].gt(0).all(axis=1)].reset_index(drop=True)
    
    if df.empty:
        return np.nan

    # 5) 숫자 피처만 선택 (객체형 제외)
    numeric_cols = set(df.select_dtypes(include=['number', 'bool']).columns)
    features_enc = [c for c in features if c in numeric_cols]
    if not features_enc:
        return np.nan

    Xv = (df[features_enc]
          .apply(pd.to_numeric, errors='coerce')
          .fillna(0)
          .to_numpy(dtype=np.float32, copy=False))

    # 6) 예측
    yhat = model.predict(Xv)
    yhat = np.asarray(yhat)
    if yhat.ndim == 1:
        yhat = yhat.reshape(-1, 1)

    if clip_to_zero:
        yhat = np.clip(yhat, 0, None)

    # 예측 컬럼 주입
    for k in range(1, 8):
        df[f'pred_tplus_{k}'] = yhat[:, k-1] if yhat.shape[1] >= k else np.nan

    # 7) sMAPE 계산 (마스크 없이, 이미 >0 행만 남음)
    scores = []
    for k in range(1, 8):
        s = weighted_smape_df(
            df,
            pred_col=f'pred_tplus_{k}',
            actual_col=f'{target_col}_tplus_{k}',
            store_col=store_col if store_col in df.columns else menu_col,
            item_col=menu_col,
            store_weights=store_weights
        )
        scores.append(s)

    return float(np.nanmean(scores))

In [10]:
def evaluate_on_valid_files(model, features, valid_files, store_col='영업장명', menu_col='영업장명_메뉴명',
                            date_col='영업일자', target_col='매출수량', store_weights=None, clip_to_zero=True):
    scores = []
    for path in valid_files:
        df_te = pd.read_csv(path)    
        use_store_col = store_col if store_col in df_te.columns else menu_col
        s = local_validate_on_test(
            df_test=df_te,
            model=model,
            features=features,
            target_col=target_col,
            menu_col=menu_col,
            store_col=use_store_col,
            date_col=date_col,
            store_weights=store_weights,
            clip_to_zero=clip_to_zero
        )
        if not np.isnan(s):
            scores.append(s)
    return float(np.mean(scores)) if scores else np.inf

In [11]:
# 경로/상수
TEST_DIR   = './data/test_pre'
TEST_GLOB  = 'TEST_*.csv'
SAMPLE_SUB = './data/sample_submission.csv'
DATE_COL   = '영업일자'
TARGET_COL = '매출수량'
MENU_COL   = '영업장명_메뉴명'
STORE_COL  = '영업장명'

train[DATE_COL] = pd.to_datetime(train[DATE_COL], errors='coerce')

# feature/target 구성
target_cols = [f'{TARGET_COL}_tplus_{i}' for i in range(1, 8)]
if not all(c in train.columns for c in target_cols):
    train = train.sort_values([MENU_COL, DATE_COL]).copy()
    for i in range(1, 8):
        col = f'{TARGET_COL}_tplus_{i}'
        shifted = train.groupby(MENU_COL)[TARGET_COL].shift(-i)
        train[col] = shifted.fillna(0)

target_cols = [f'{TARGET_COL}_tplus_{i}' for i in range(1, 8)]
exclude     = set([DATE_COL, TARGET_COL] + target_cols)

train = train.loc[train[target_cols].gt(0).all(axis=1)].reset_index(drop=True)

numeric_cols = set(train.select_dtypes(include=['number', 'bool']).columns)
features = [c for c in train.columns if (c in numeric_cols) and (c not in exclude)]

X_train = train[features].values
Y_train = train[target_cols].values

# 튜닝 검증 세트(파일)
import glob, os, re
all_test_files = sorted(glob.glob(os.path.join(TEST_DIR, TEST_GLOB)))
valid_files = all_test_files[-2:] if len(all_test_files) >= 2 else all_test_files
print('[INFO] valid files for tuning:', [os.path.basename(p) for p in valid_files])

[INFO] valid files for tuning: ['TEST_08.csv', 'TEST_09.csv']


## Optuna Objective (XGB)

In [13]:
def make_objective_lgbm(X_train, Y_train, train_df, features, valid_files,
                        tune_store_weights=True):

    store_list = sorted(map(str, train_df[STORE_COL].astype(str).unique()))
    focus = {'담하', '미라시아'}  # 이 둘은 '다른 업장'보다 크게 샘플링

    def objective(trial):
        # ---- LightGBM 하이퍼파라미터 (sMAPE와 정렬 고려 → L1 선호) ----
        params = {
            'objective'              : 'l1',     # (= MAE), 'l2'면 RMSE
            'metric'                 : 'mae',
            'n_estimators'           : trial.suggest_int('n_estimators', 400, 1200, step=100),
            'learning_rate'          : trial.suggest_float('learning_rate', 0.02, 0.2, log=True),
            'num_leaves'             : trial.suggest_int('num_leaves', 31, 255),
            'max_depth'              : trial.suggest_int('max_depth', -1, 12),  # -1 = no limit
            'min_child_samples'      : trial.suggest_int('min_child_samples', 10, 100),
            'min_sum_hessian_in_leaf': trial.suggest_float('min_sum_hessian_in_leaf', 1e-3, 10.0, log=True),
            'feature_fraction'       : trial.suggest_float('feature_fraction', 0.6, 1.0),  # = colsample_bytree
            'bagging_fraction'       : trial.suggest_float('bagging_fraction', 0.6, 1.0),  # = subsample
            'bagging_freq'           : 1,
            'lambda_l1'              : trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
            'lambda_l2'              : trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
            'random_state'           : 42,
            'n_jobs'                 : -1,
        }

        # ---- 업장별 가중치(선택적으로 튜닝) ----
        store_weights = None
        if tune_store_weights:
            store_weights = {}
            for st in store_list:
                if st in focus:
                    # 포커스 업장은 상대적으로 높게
                    store_weights[st] = trial.suggest_float(f'w_{st}', 1.5, 5.0)
                else:
                    store_weights[st] = trial.suggest_float(f'w_{st}', 0.6, 1.5)

        # 행단위 샘플가중치 (정규화=True: 평균 1로 맞춰 학습률 안정화)
        sample_w = make_store_train_weight_from_series(
            train_df[STORE_COL], store_weights, normalize=True
        )

        # ---- 모델 학습 (다중 타깃: 7일) ----
        base  = LGBMRegressor(**params)
        model = MultiOutputRegressor(base)
        model.fit(X_train, Y_train, sample_weight=sample_w)

        # ---- 검증 (동일 store_weights로 가중 sMAPE) ----
        score = evaluate_on_valid_files(
            model, features, valid_files,
            store_col=STORE_COL, menu_col=MENU_COL, date_col=DATE_COL, target_col=TARGET_COL,
            store_weights=store_weights, clip_to_zero=True
        )
        return score

    return objective

In [14]:
# === 카테고리 인코딩 (XGB/LGBM용) ===
# (튜닝/학습 셀 위에 두세요)
import re
# 1) 카테고리 후보 (존재하는 것만 사용)
cand_cat = ['영업장명_메뉴명','영업장명','메뉴명','dow_month']
cat_cols = [c for c in cand_cat if c in train.columns]
# object dtype 컬럼도 추가 (중복 제거)
cat_cols += [c for c in train.select_dtypes('object').columns if c not in cat_cols]
cat_cols = list(dict.fromkeys(cat_cols))

print('[INFO] categorical columns:', cat_cols)

# 2) (train + 모든 test_pre) 값의 합집합으로 카테고리 맵 생성 → unknown은 -1 처리
import glob, os

def build_cat_maps(train_df, test_dir=TEST_DIR, pattern=TEST_GLOB):
    values = {c:set(train_df[c].dropna().astype(str).unique()) for c in cat_cols}
    for path in glob.glob(os.path.join(test_dir, pattern)):
        te = pd.read_csv(path)
        te = feature_engineering(te)
        for c in cat_cols:
            if c in te.columns:
                values[c].update(te[c].dropna().astype(str).unique())
    # 안정적인 정렬(옵션): 정렬/미정렬 아무거나 사용 가능
    maps = {c:{v:i for i, v in enumerate(sorted(values[c]))} for c in cat_cols}
    return maps

cat_maps = build_cat_maps(train)

def apply_cat_maps(df, maps):
    df = df.copy()
    for c, m in maps.items():
        if c in df.columns:
            df[c] = df[c].astype(str).map(m).fillna(-1).astype(int)
    return df

# 3) 학습 데이터 인코딩 후 X/Y 재생성
train_enc = apply_cat_maps(train, cat_maps)

# 타깃 파생 접두어
forbidden_prefix = f'{TARGET_COL}_tplus_'

# 인코딩된 train_enc 기준으로 features 재구성
features = [
    c for c in train_enc.columns
    if (c not in [TARGET_COL, DATE_COL])      # 원 타깃/날짜 제외 
    and (train_enc[c].dtype != 'O')            # object 제외
]

pattern = re.compile(rf'^{TARGET_COL}_tplus_\d+$')  # 숫자로 끝나는 경우만 금지
features = [
    c for c in train.columns
    if c not in [TARGET_COL, DATE_COL] and not pattern.match(c)
]

# 학습 데이터 다시 만들기
X_train = train_enc[features].values
target_cols = [f'{TARGET_COL}_tplus_{i}' for i in range(1, 8)]
Y_train = train_enc[target_cols].values

print('[INFO] len(features)=', len(features))
assert not any(
    c.startswith(forbidden_prefix) and not c.endswith('_mask')
    for c in features
)

[INFO] categorical columns: ['영업장명_메뉴명', '영업장명', '메뉴명']
[INFO] len(features)= 66


In [15]:
train.to_csv('./data/train_.csv', index=False, encoding='utf-8-sig')

## Run Study

In [18]:
study = optuna.create_study(direction='minimize')
study.optimize(
    make_objective_lgbm(X_train, Y_train, train, features, valid_files, tune_store_weights=True),
    n_trials=50, show_progress_bar=False
)

[I 2025-08-23 23:58:38,742] A new study created in memory with name: no-name-b2899559-1b00-4242-80c4-8f38387e6512
[W 2025-08-23 23:58:38,746] Trial 0 failed with parameters: {'n_estimators': 900, 'learning_rate': 0.0779713364388696, 'num_leaves': 213, 'max_depth': 0, 'min_child_samples': 17, 'min_sum_hessian_in_leaf': 0.8640073444294372, 'feature_fraction': 0.701782114181222, 'bagging_fraction': 0.7130423908058703, 'lambda_l1': 0.002478406747420697, 'lambda_l2': 0.10824324676595726, 'w_느티나무': 0.8029863076338754, 'w_담하': 2.906504962667176, 'w_라그로타': 1.0459395238602125, 'w_미라시아': 1.7431622900778736, 'w_연회장': 1.2698391823700215, 'w_카페테리아': 1.0951141382810228, 'w_포레스트릿': 0.9431116782526675, 'w_화담숲주막': 0.9497985566910633, 'w_화담숲카페': 0.6092229863040742} because of the following error: NameError("name 'make_store_train_weight_from_series' is not defined").
Traceback (most recent call last):
  File "C:\Users\user\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
 

NameError: name 'make_store_train_weight_from_series' is not defined

In [ ]:
best_params = {k: v for k, v in study.best_params.items() if not k.startswith('w_')}
best_store_weights = {k.replace('w_', ''): v for k, v in study.best_params.items() if k.startswith('w_')}

print('[BEST LGBM hp]', best_params)
print('[BEST store weights]', best_store_weights)
print('[BEST sMAPE]', study.best_value)

In [ ]:
final_base  = LGBMRegressor(
    objective='regression', metric='mae', # or rmse
    random_state=42, n_jobs=-1,
    **best_params
)
final_lgbm = MultiOutputRegressor(final_base)

final_sample_w = make_store_train_weight_from_series(
    train[STORE_COL], best_store_weights if best_store_weights else None, normalize=True
)
final_lgbm.fit(X_train, Y_train, sample_weight=final_sample_w)

## Predict & Build Submission

In [ ]:
all_rows = []
for path in all_test_files:
    df_te = pd.read_csv(path)

    # 1) FE
    df_te = feature_engineering(df_te)
    # 원본 메뉴명 보존
    df_te['menu_orig'] = df_te[MENU_COL].astype(str)

    # 2) 인코딩
    df_te_enc = apply_cat_maps(df_te, cat_maps)
    # 인코딩 후에도 원본 메뉴명 열 유지
    df_te_enc['menu_orig'] = df_te['menu_orig'].values

    # 3) 피처 보강/숫자화
    for col in features:
        if col not in df_te_enc.columns:
            df_te_enc[col] = 0
    df_te_enc[features] = df_te_enc[features].apply(pd.to_numeric, errors='coerce').fillna(0).astype('float32')

    # 4) 날짜 처리
    if DATE_COL in df_te_enc.columns:
        df_te_enc[DATE_COL] = pd.to_datetime(df_te_enc[DATE_COL], errors='coerce')

    fname = os.path.basename(path)
    m = re.search(r'(TEST_\d+)', fname); test_prefix = m.group(1) if m else 'TEST_??'

    # ▶ 그룹바이 기준을 'menu_orig'(문자열)로!
    for menu_name, sub in df_te_enc.groupby('menu_orig'):
        sub = sub.sort_values(DATE_COL) if DATE_COL in sub.columns else sub
        X_last = sub[features].tail(1).to_numpy(dtype=np.float32, copy=False)
        yhat = final_xgb.predict(X_last).ravel()
        yhat = np.clip(yhat, 0, None)

        for k in range(7):
            all_rows.append({
                '영업일자': f'{test_prefix}+{k+1}일',
                '영업장명_메뉴명': menu_name,   # ← 문자열 이름으로 저장
                '매출수량': float(yhat[k])
            })

full_pred_df = pd.DataFrame(all_rows, columns=['영업일자','영업장명_메뉴명','매출수량'])

In [ ]:
full_pred_df

In [ ]:
full_pred_df.to_csv('./data/lgbm_1.csv', index=False, encoding='utf-8-sig')

# Submission

In [ ]:
def convert_to_submission_format(pred_df: pd.DataFrame, sample_submission: pd.DataFrame):
    pred_dict = dict(zip(
        zip(pred_df['영업일자'], pred_df['영업장명_메뉴명']),
        pred_df['매출수량']
    ))
    final_df = sample_submission.copy()
    for row_idx in final_df.index:
        date = final_df.loc[row_idx, '영업일자']
        for col in final_df.columns[1:]:
            final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
    return final_df

In [ ]:
sample_submission = pd.read_csv(SAMPLE_SUB)
submission = convert_to_submission_format(full_pred_df, sample_submission)
submission.head()

In [ ]:
submission.to_csv('Gonjiam_submission_lgbm_1.csv', index=False, encoding='utf-8-sig')